# Distillation RL-Assisted MPC Combined Supervisor

This notebook keeps the unified four-agent combined workflow, while aligning the default setpoints, run lengths, and decision interval with the archived distillation combined notebook where that mapping exists. It remains more capable than the legacy combined notebook, but the top-level controls are organized to feel closer to the original distillation study notebooks.

In [ ]:
from pathlib import Path
import os

from utils.notebook_setup import prepare_distillation_notebook_env, print_notebook_summary

# Main notebook controls.
# The defaults here follow the archived distillation combined notebook where possible, while keeping the unified four-agent supervisor available.
RUN_MODE = "nominal"  # "nominal" | "disturb"
DISTURBANCE_PROFILE = "none"  # "none" | "ramp" | "fluctuation"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

ENABLE_HORIZON = True
HORIZON_STATE_MODE = "standard"

ENABLE_MATRIX = True
MATRIX_AGENT_KIND = "td3"
MATRIX_STATE_MODE = "standard"

ENABLE_WEIGHTS = True
WEIGHTS_AGENT_KIND = "td3"
WEIGHTS_STATE_MODE = "standard"

ENABLE_RESIDUAL = True
RESIDUAL_AGENT_KIND = "td3"
RESIDUAL_STATE_MODE = "standard"
USE_RHO_AUTHORITY = True

ASPEN_PRESET = "default"
ASPEN_PATH_OVERRIDE = None
SNAPS_PATH_OVERRIDE = None
ASPEN_ROOT_OVERRIDE = None
DISTILLATION_VISIBLE = True

DISTILLATION_DATA_DIR_OVERRIDE = None
DISTILLATION_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides.
RESULT_PREFIX_OVERRIDE = None
COMPARE_PREFIX_OVERRIDE = None
BASELINE_MPC_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to use the archived defaults.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
WARM_START_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None
COMPARE_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(
    run_mode=RUN_MODE,
    disturbance_profile=DISTURBANCE_PROFILE,
    family="combined",
    aspen_preset=ASPEN_PRESET,
    dyn_path_override=ASPEN_PATH_OVERRIDE,
    snaps_path_override=SNAPS_PATH_OVERRIDE,
    aspen_root_override=ASPEN_ROOT_OVERRIDE,
    data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE,
    results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

print_notebook_summary(
    "Distillation combined notebook configuration",
    {
        "Distillation data dir": DATA_DIR,
        "Distillation results": RESULT_DIR,
        "Run mode": RUN_MODE,
        "Disturbance profile": DISTURBANCE_PROFILE,
        "Enable horizon": ENABLE_HORIZON,
        "Enable matrix": ENABLE_MATRIX,
        "Enable weights": ENABLE_WEIGHTS,
        "Enable residual": ENABLE_RESIDUAL,
        "Aspen source": ASPEN_SOURCE,
        "Aspen dynf": DYN_PATH,
        "Aspen snaps": SNAPS_PATH,
    },
)

def build_combined_suffix() -> str:
    parts = []
    if ENABLE_HORIZON:
        parts.append(f"h_dqn_{HORIZON_STATE_MODE}")
    if ENABLE_MATRIX:
        parts.append(f"m_{MATRIX_AGENT_KIND}_{MATRIX_STATE_MODE}")
    if ENABLE_WEIGHTS:
        parts.append(f"w_{WEIGHTS_AGENT_KIND}_{WEIGHTS_STATE_MODE}")
    if ENABLE_RESIDUAL:
        rho_suffix = "rho" if USE_RHO_AUTHORITY else "no_rho"
        parts.append(f"r_{RESIDUAL_AGENT_KIND}_{RESIDUAL_STATE_MODE}_{rho_suffix}")
    return "__".join(parts) if parts else "baseline"

import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from SACAgent.sac_agent import SACAgent
from systems.distillation import DISTILLATION_SYSTEM_METADATA
from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_COMBINED_RUN_PROFILES,
    DISTILLATION_COMBINED_SETPOINTS_PHYS,
    DISTILLATION_INPUT_BOUNDS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_OBSERVER_POLES,
    DISTILLATION_SETPOINT_RANGE_PHYS,
    DISTILLATION_SS_INPUTS,
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    MATRIX_MULTIPLIER_BOUNDS,
    RESIDUAL_BOUNDS,
    RL_REWARD_DEFAULTS,
    WEIGHT_MULTIPLIER_BOUNDS,
)
from systems.distillation.data_io import canonical_baseline_path, load_distillation_system_data
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule
from Simulation.mpc import MpcSolverGeneral
from TD3Agent.agent import TD3Agent
from utils.combined_runner import run_combined_supervisor
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.observer import compute_observer_gain
from utils.plotting import plot_combined_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim, default_mismatch_scale


In [ ]:
RUN_PROFILE = DISTILLATION_COMBINED_RUN_PROFILES[(RUN_MODE, DISTURBANCE_PROFILE)]


nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS.copy()
u_min = DISTILLATION_INPUT_BOUNDS["u_min"].copy()
u_max = DISTILLATION_INPUT_BOUNDS["u_max"].copy()
setpoint_y = DISTILLATION_SETPOINT_RANGE_PHYS.copy()

# The legacy combined notebook used the combined setpoint pair below.
y_sp_scenario_phys = DISTILLATION_COMBINED_SETPOINTS_PHYS.copy()
system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=DELTA_T_HOURS,
    visible=DISTILLATION_VISIBLE,
)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])

ACTIVE_AGENT_TAGS = []
if ENABLE_HORIZON:
    ACTIVE_AGENT_TAGS.append(f"h-{HORIZON_STATE_MODE}")
if ENABLE_MATRIX:
    ACTIVE_AGENT_TAGS.append(f"m-{MATRIX_AGENT_KIND}-{MATRIX_STATE_MODE}")
if ENABLE_WEIGHTS:
    ACTIVE_AGENT_TAGS.append(f"w-{WEIGHTS_AGENT_KIND}-{WEIGHTS_STATE_MODE}")
if ENABLE_RESIDUAL:
    ACTIVE_AGENT_TAGS.append(f"r-{RESIDUAL_AGENT_KIND}-{RESIDUAL_STATE_MODE}-{'rho' if USE_RHO_AUTHORITY else 'no_rho'}")
ACTIVE_AGENT_SUFFIX = "__".join(ACTIVE_AGENT_TAGS) if ACTIVE_AGENT_TAGS else "no_agents"
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or f"distillation_combined_{RUN_MODE}_{DISTURBANCE_PROFILE}_{ACTIVE_AGENT_SUFFIX}_unified"
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or f"distillation_compare_combined_{RUN_MODE}_{DISTURBANCE_PROFILE}_{ACTIVE_AGENT_SUFFIX}"
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE)
n_tests = RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
warm_start = RUN_PROFILE["warm_start"] if WARM_START_OVERRIDE is None else int(WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE["test_cycle"]) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else int(COMPARE_START_EPISODE_OVERRIDE)
TOTAL_STEPS = n_tests * set_points_len * len(y_sp_scenario_phys)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS)


print_notebook_summary(
    "Resolved experiment parameters",
    {
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "plot_start_episode": PLOT_START_EPISODE,
        "compare_start_episode": COMPARE_START_EPISODE,
        "legacy_combined_setpoints_phys": y_sp_scenario_phys.tolist(),
        "baseline_mpc_path": BASELINE_MPC_PATH,
    },
)


In [ ]:
# User-editable controller defaults. Edit these values directly when you want to change controller/agent behavior.
poles = DISTILLATION_OBSERVER_POLES.copy()
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
MISMATCH_SCALE = default_mismatch_scale(min_max_dict)
MISMATCH_CLIP = 3.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ACTOR_FREEZE = 0
HORIZON_EXPLORATION_MODE = "epsilon"
HORIZON_LOSS_TYPE = "huber"
TD3_EXPLORATION_MODE = "param_noise"
TD3_LOSS_TYPE = "huber"
SAC_LOSS_TYPE = "huber"
USE_SHIFTED_MPC_WARM_START = False  # False matches legacy zero initialization; set True to reuse the previous MPC sequence.
# The legacy combined notebook changed the horizon every 5 steps.
DECISION_INTERVAL = 5
TD3_PARAM_NOISE_RESAMPLE_INTERVAL = DECISION_INTERVAL

PREDICT_GRID = HORIZON_PREDICT_GRID
CONTROL_GRID = HORIZON_CONTROL_GRID
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
predict_h = 6
cont_h = 3

Q1_penalty = 1.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
MODEL_LOW = MATRIX_MULTIPLIER_BOUNDS[MATRIX_AGENT_KIND]["low"].copy()
MODEL_HIGH = MATRIX_MULTIPLIER_BOUNDS[MATRIX_AGENT_KIND]["high"].copy()
WEIGHTS_LOW = WEIGHT_MULTIPLIER_BOUNDS["low"].copy()
WEIGHTS_HIGH = WEIGHT_MULTIPLIER_BOUNDS["high"].copy()
RESIDUAL_LOW = RESIDUAL_BOUNDS["low"].copy()
RESIDUAL_HIGH = RESIDUAL_BOUNDS["high"].copy()

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    N_INPUTS,
    **RL_REWARD_DEFAULTS,
)
print(reward_params)

nominal_qi = 0.0
nominal_qs = 0.0
nominal_hA = 0.0
qi_change = 1.0
qs_change = 1.0
ha_change = 1.0



def make_continuous_agent(agent_kind, state_dim, action_dim):
    if agent_kind == "td3":
        return TD3Agent(
            state_dim=state_dim,
            action_dim=action_dim,
            actor_hidden=[512, 512, 512],
            critic_hidden=[512, 512, 512],
            gamma=0.995,
            actor_lr=1e-4,
            critic_lr=1e-4,
            batch_size=128,
            policy_delay=4,
            target_policy_smoothing_noise_std=0.1,
            noise_clip=0.2,
            max_action=1.0,
            tau=0.005,
            std_start=0.2,
            std_end=0.02,
            std_decay_rate=0.99995,
            std_decay_mode="exp",
            buffer_size=25_000,
            device=DEVICE,
            actor_freeze=ACTOR_FREEZE,
            loss_type=SAC_LOSS_TYPE,
        )
    if agent_kind == "sac":
        return SACAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            actor_hidden=[512, 512, 512],
            critic_hidden=[512, 512, 512],
            gamma=0.995,
            actor_lr=1e-4,
            critic_lr=1e-4,
            alpha_lr=1e-4,
            batch_size=128,
            grad_clip_norm=10.0,
            init_alpha=0.01,
            learn_alpha=True,
            target_entropy=-action_dim,
            target_update="soft",
            tau=0.005,
            hard_update_interval=10_000,
            activation="relu",
            use_layernorm=False,
            dropout=0.0,
            max_action=1.0,
            buffer_size=20_000,
            use_per=True,
            device=DEVICE,
            use_adamw=True,
            actor_freeze=ACTOR_FREEZE,
            exploration_mode=TD3_EXPLORATION_MODE,
            loss_type=TD3_LOSS_TYPE,
            param_noise_resample_interval=TD3_PARAM_NOISE_RESAMPLE_INTERVAL,
        )
    raise ValueError("Continuous agent kind must be 'td3' or 'sac'.")


horizon_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, HORIZON_STATE_MODE)
matrix_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, MATRIX_STATE_MODE)
weights_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, WEIGHTS_STATE_MODE)
residual_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, RESIDUAL_STATE_MODE)

horizon_agent = None
if ENABLE_HORIZON:
    horizon_agent = DQNAgent(
        state_dim=horizon_state_dim,
        action_dim=len(HORIZON_RECIPES),
        hidden_dim=[512, 512, 512, 512, 512],
        gamma=0.99,
        lr=1e-4,
        batch_size=128,
        buffer_size=40_000,
        grad_clip_norm=10.0,
        double_dqn=True,
        target_update="soft",
        tau=0.01,
        hard_update_interval=10_000,
        target_combine="q1",
        activation="relu",
        use_layer_norm=False,
        dropout=0.0,
        device=DEVICE,
        exploration_mode=HORIZON_EXPLORATION_MODE,
        loss_type=HORIZON_LOSS_TYPE,
        eps_start=0.3,
        eps_end=0.01,
        eps_decay_rate=0.99999,
        eps_decay_mode="exp",
    )

matrix_agent = make_continuous_agent(MATRIX_AGENT_KIND, matrix_state_dim, 1 + N_INPUTS) if ENABLE_MATRIX else None
weights_agent = make_continuous_agent(WEIGHTS_AGENT_KIND, weights_state_dim, 4) if ENABLE_WEIGHTS else None
residual_agent = make_continuous_agent(RESIDUAL_AGENT_KIND, residual_state_dim, N_INPUTS) if ENABLE_RESIDUAL else None

print(f"Using device: {DEVICE}")
print(f"Enabled agents: horizon={ENABLE_HORIZON}, matrix={ENABLE_MATRIX}, weights={ENABLE_WEIGHTS}, residual={ENABLE_RESIDUAL}")
print(f"State dimensions | horizon={horizon_state_dim}, matrix={matrix_state_dim}, weights={weights_state_dim}, residual={residual_state_dim}")


In [ ]:
combined_cfg = {
    "run_mode": RUN_MODE,
    "notebook_source": "distillation_RL_assisted_MPC_combined_unified.ipynb",
    "decision_interval": DECISION_INTERVAL,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "horizon_cfg": {
        "enabled": ENABLE_HORIZON,
        "state_mode": HORIZON_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "horizon_recipes": HORIZON_RECIPES,
        "default_horizons": (predict_h, cont_h),
    },
    "matrix_cfg": {
        "enabled": ENABLE_MATRIX,
        "agent_kind": MATRIX_AGENT_KIND,
        "state_mode": MATRIX_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "low_coef": MODEL_LOW,
        "high_coef": MODEL_HIGH,
    },
    "weight_cfg": {
        "enabled": ENABLE_WEIGHTS,
        "agent_kind": WEIGHTS_AGENT_KIND,
        "state_mode": WEIGHTS_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "low_coef": WEIGHTS_LOW,
        "high_coef": WEIGHTS_HIGH,
    },
    "residual_cfg": {
        "enabled": ENABLE_RESIDUAL,
        "agent_kind": RESIDUAL_AGENT_KIND,
        "state_mode": RESIDUAL_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "use_rho_authority": USE_RHO_AUTHORITY,
        "low_coef": RESIDUAL_LOW,
        "high_coef": RESIDUAL_HIGH,
    },
}

agents = {}
if ENABLE_HORIZON:
    agents["horizon"] = horizon_agent
if ENABLE_MATRIX:
    agents["matrix"] = matrix_agent
if ENABLE_WEIGHTS:
    agents["weights"] = weights_agent
if ENABLE_RESIDUAL:
    agents["residual"] = residual_agent

runtime_ctx = {
    "system": system,
    "agents": agents,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
}

result_bundle = run_combined_supervisor(combined_cfg=combined_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH


In [ ]:
out_dir_rl = plot_combined_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "reward_fn": reward_fn,
        "system_metadata": DISTILLATION_SYSTEM_METADATA,
        "include_baseline_compare": True,
        "compare_mode": RUN_MODE,
        "compare_prefix": COMPARE_PREFIX,
        "compare_directory": RESULT_DIR,
        "mpc_path_or_dir": BASELINE_MPC_PATH,
        "compare_start_episode": COMPARE_START_EPISODE,
    },
)

print("Combined result directory:", out_dir_rl)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass
